# 🎓 Fine-Tuning DeBERTa-v3 for Multi-Criteria IELTS Essay Scoring

This notebook trains an **encoder-only transformer** (`microsoft/deberta-v3-small` or `base`) to predict all **4 official IELTS criteria** + **Overall Band Score** simultaneously:
1. **Task Response (TR)**
2. **Coherence & Cohesion (CC)**
3. **Lexical Resource (LR)**
4. **Grammatical Range & Accuracy (GRA)**
5. **Overall Band Score**

In [ ]:
# 1. Verify GPU availability
import torch
print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    print(f"✅ GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! In Colab: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 2. Install required libraries
!pip install -q transformers accelerate datasets scikit-learn pandas

In [ ]:
# 3. Clone / Pull IELTS_Copilot repository
import os
if not os.path.exists("IELTS_Copilot"):
    !git clone https://github.com/hamzayevtemur-lab/IELTS_Copilot.git
    %cd IELTS_Copilot
else:
    %cd IELTS_Copilot
    !git pull

In [ ]:
# 4. Inspect training dataset shape and target columns
import pandas as pd
train_df = pd.read_csv("data/processed/train_essays.csv")
print(f"Training essays count: {len(train_df)}")
train_df[["question", "overall_band", "task_achievement_band", "coherence_cohesion_band", "lexical_resource_band", "grammatical_range_band"]].head(3)

In [ ]:
# 5. Launch Multi-Criteria Fine-Tuning
from src.scoring.train import train

trainer = train(
    train_csv="data/processed/train_essays.csv",
    val_csv="data/processed/val_essays.csv",
    model_name="microsoft/deberta-v3-small",  # or 'microsoft/deberta-v3-base'
    output_dir="checkpoints/deberta_ielts_scorer",
    batch_size=8,      # Increase to 16 if using A100 or T4 with 16GB
    num_epochs=4,      # 4 epochs provides solid convergence
    learning_rate=2e-5,
    max_length=512,
)

In [ ]:
# 6. Evaluate on Unseen Test Essays
import torch
from transformers import AutoTokenizer
from src.scoring.model import DebertaIELTSScorer
from src.scoring.dataset import round_to_ielts_band, TARGET_COLUMNS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
saved_path = "checkpoints/deberta_ielts_scorer/final_model"

# Load fine-tuned weights
eval_model = DebertaIELTSScorer("microsoft/deberta-v3-small").to(device)
eval_model.load_state_dict(torch.load(f"{saved_path}/pytorch_model.bin", map_location=device))
eval_model.eval()

eval_tok = AutoTokenizer.from_pretrained(saved_path)

# Test on 3 unseen test essays
test_df = pd.read_csv("data/processed/test_essays.csv").head(3)
print("=" * 70)
print("🔍 PREDICTIONS ON UNSEEN ESSAYS")
print("=" * 70)

for i, row in test_df.iterrows():
    q = row["question"]
    essay = row["essay_text"]
    inputs = eval_tok(q, essay, max_length=512, truncation=True, padding="max_length", return_tensors="pt").to(device)
    
    with torch.no_grad():
        preds = eval_model(inputs["input_ids"], inputs["attention_mask"])["logits"][0].cpu().numpy()
        
    rounded = round_to_ielts_band(preds)
    
    print(f"\nSample #{i+1}: {q[:70]}...")
    print("Criterion | Actual Examiner Band | Model Predicted Band (Raw)")
    print("-" * 60)
    criteria = ["TR", "CC", "LR", "GRA", "Overall"]
    actuals = [row["task_achievement_band"], row["coherence_cohesion_band"], row["lexical_resource_band"], row["grammatical_range_band"], row["overall_band"]]
    for c, act, pr, r_pr in zip(criteria, actuals, preds, rounded):
        print(f"{c:9} | {float(act):20.1f} | {r_pr:5.1f}  (raw: {pr:.2f})")

In [ ]:
# 7. Compress trained model for local download
!zip -r deberta_ielts_scorer.zip checkpoints/deberta_ielts_scorer/final_model
print("✅ Model zipped! You can download 'deberta_ielts_scorer.zip' from Colab's file browser to your local Mac.")